<a href="https://colab.research.google.com/github/Manjot-Coder/IP_LabAssignments/blob/main/Assignment-10/Q1_Q2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#Q1

import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from google.colab import files
import io

# ---------------------------------------
# STEP 1: Upload grayscale image
# ---------------------------------------

uploaded = files.upload()
filename = next(iter(uploaded))

img = Image.open(io.BytesIO(uploaded[filename])).convert("L")
image = np.array(img, dtype=np.uint8)

# ---------------------------------------
# STEP 2: Take inputs
# ---------------------------------------

initial_size = int(input("Enter initial window size (odd, e.g. 3): "))
Smax = int(input("Enter maximum window size Smax (odd, e.g. 7): "))
noise_density = float(input("Enter salt and pepper noise density (e.g. 0.05): "))

if (initial_size < 3 or initial_size % 2 == 0 or
    Smax < initial_size or Smax % 2 == 0 or
    not 0 <= noise_density <= 1):
    raise ValueError("Enter valid odd window sizes and noise density.")

# ---------------------------------------
# STEP 3: Add salt and pepper noise
# ---------------------------------------

rng = np.random.default_rng()

noisy_image = image.copy()
random_values = rng.random(image.shape)

noisy_image[random_values < noise_density / 2] = 0
noisy_image[
    (random_values >= noise_density / 2) &
    (random_values < noise_density)
] = 255


# ---------------------------------------
# STEP 4: Normal Median Filter
# ---------------------------------------

def median_filter(img, size):

    pad = size // 2
    padded = np.pad(img, pad, mode="edge")

    result = np.zeros(img.shape, dtype=np.uint8)

    for i in range(img.shape[0]):
        for j in range(img.shape[1]):

            region = padded[i:i+size, j:j+size]

            result[i, j] = np.median(region)

    return result


median_result = median_filter(noisy_image, initial_size)


# ---------------------------------------
# STEP 5: Adaptive Median Filter
# ---------------------------------------

def adaptive_median_filter(img, initial_size, Smax):

    height, width = img.shape
    result = np.zeros((height, width), dtype=np.uint8)

    max_pad = Smax // 2
    padded = np.pad(img, max_pad, mode="edge")

    for i in range(height):
        for j in range(width):

            window_size = initial_size
            output = None

            while window_size <= Smax:

                pad = window_size // 2

                region = padded[
                    i + max_pad - pad : i + max_pad + pad + 1,
                    j + max_pad - pad : j + max_pad + pad + 1
                ]

                zmin = np.min(region)
                zmax = np.max(region)
                zmed = np.median(region)
                zxy = padded[i + max_pad, j + max_pad]

                # Stage A
                A1 = zmed - zmin
                A2 = zmed - zmax

                if A1 > 0 and A2 < 0:

                    # Stage B
                    B1 = zxy - zmin
                    B2 = zxy - zmax

                    if B1 > 0 and B2 < 0:
                        output = zxy
                    else:
                        output = zmed

                    break

                else:
                    window_size += 2

            # If maximum window size is reached
            if output is None:
                output = zmed

            result[i, j] = output

    return result


adaptive_result = adaptive_median_filter(
    noisy_image, initial_size, Smax
)


# ---------------------------------------
# STEP 6: Calculate PSNR
# ---------------------------------------

def calculate_psnr(original, filtered):

    mse = np.mean(
        (original.astype(float) - filtered.astype(float)) ** 2
    )

    if mse == 0:
        return float("inf")

    psnr = 10 * np.log10((255 ** 2) / mse)

    return psnr


psnr_noisy = calculate_psnr(image, noisy_image)
psnr_median = calculate_psnr(image, median_result)
psnr_adaptive = calculate_psnr(image, adaptive_result)


# ---------------------------------------
# STEP 7: Display results
# ---------------------------------------

images = [
    image,
    noisy_image,
    median_result,
    adaptive_result
]

titles = [
    "Original Image",
    "Noisy Image",
    "Median Filter",
    "Adaptive Median Filter"
]

plt.figure(figsize=(14, 5))

for i in range(4):

    plt.subplot(1, 4, i+1)
    plt.imshow(images[i], cmap="gray")
    plt.title(titles[i])
    plt.axis("off")

plt.tight_layout()
plt.show()


# ---------------------------------------
# STEP 8: Display PSNR comparison
# ---------------------------------------

print("PSNR Comparison")
print("-----------------------------")
print(f"Noisy Image PSNR: {psnr_noisy:.2f} dB")
print(f"Median Filter PSNR: {psnr_median:.2f} dB")
print(f"Adaptive Median PSNR: {psnr_adaptive:.2f} dB")

if psnr_adaptive > psnr_median:
    print("Adaptive Median Filter performs better.")
elif psnr_median > psnr_adaptive:
    print("Median Filter performs better.")
else:
    print("Both filters have equal PSNR.")

# ---------------------------------------
# STEP 9: Save output images
# ---------------------------------------

Image.fromarray(noisy_image).save("noisy_image.jpg")
Image.fromarray(median_result).save("median_filter.jpg")
Image.fromarray(adaptive_result).save("adaptive_median_filter.jpg")

print("All images saved successfully!")


In [2]:
#Q2

import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from google.colab import files
import io

# ---------------------------------------
# STEP 1: Upload grayscale image
# ---------------------------------------

uploaded = files.upload()
filename = next(iter(uploaded))

img = Image.open(io.BytesIO(uploaded[filename])).convert("L")
image = np.array(img, dtype=np.uint8)


# ---------------------------------------
# STEP 2: Calculate histogram
# ---------------------------------------

histogram = np.bincount(
    image.flatten(), minlength=256
)

total_pixels = image.size

# Normalize histogram
probability = histogram / total_pixels

# ---------------------------------------
# STEP 3: Calculate total mean intensity
# ---------------------------------------

intensity = np.arange(256)

total_mean = np.sum(
    intensity * probability
)

# ---------------------------------------
# STEP 4: Find Otsu threshold
# ---------------------------------------

max_variance = 0
best_threshold = 0

weight_background = 0
sum_background = 0

for t in range(256):

    weight_background += probability[t]
    sum_background += t * probability[t]

    weight_foreground = 1 - weight_background

    if weight_background == 0 or weight_foreground == 0:
        continue

    mean_background = sum_background / weight_background

    mean_foreground = (
        total_mean - sum_background
    ) / weight_foreground

    # Between-class variance
    variance = (
        weight_background *
        weight_foreground *
        (mean_background - mean_foreground) ** 2
    )

    if variance > max_variance:

        max_variance = variance
        best_threshold = t


# ---------------------------------------
# STEP 5: Apply threshold
# ---------------------------------------

threshold_image = np.zeros(image.shape, dtype=np.uint8)

threshold_image[image > best_threshold] = 255

# ---------------------------------------
# STEP 6: Display results
# ---------------------------------------

plt.figure(figsize=(14, 5))

plt.subplot(1, 3, 1)
plt.imshow(image, cmap="gray")
plt.title("Original Grayscale Image")
plt.axis("off")

plt.subplot(1, 3, 2)
plt.plot(histogram)
plt.axvline(
    best_threshold,
    color="red",
    linestyle="--",
    label=f"Threshold = {best_threshold}"
)
plt.title("Image Histogram")
plt.xlabel("Pixel Intensity")
plt.ylabel("Frequency")
plt.legend()

plt.subplot(1, 3, 3)
plt.imshow(threshold_image, cmap="gray")
plt.title("Otsu Segmented Image")
plt.axis("off")

plt.tight_layout()
plt.show()


# ---------------------------------------
# STEP 7: Print threshold and save image
# ---------------------------------------

print("Optimal Otsu Threshold:", best_threshold)

Image.fromarray(threshold_image).save(
    "otsu_segmented_image.jpg"
)

print("Segmented image saved successfully!")


KeyboardInterrupt: 